In [1]:
import pandas as pd  # Para manipulación de datos
import matplotlib.pyplot as plt  # Para visualización
import seaborn as sns  # Para gráficos más estilizados
import numpy as np
import os

In [2]:
# Cargar los datos
csv_file_path = r"../../data/nuevos/cleaned_data_nuevos.csv"
df = pd.read_csv(csv_file_path)

df.head()

,timestamp,open_binance,high_binance,low_binance,close_binance,volumen_binance,trades_binance,var_diaria_binance,rango_dia_binance,volatilidad_50_binance,...,momentum_200_dxy,SMA_50_dxy,EMA_50_dxy,SMA_200_dxy,EMA_200_dxy,RSI_dxy,pct_cambio_7d,tendencia_futura_7d,pct_cambio_30d,tendencia_futura_30d
0,2023-02-25,23184.04,23219.13,22722.00,23157.07,191311.81010,9124568.0,-0.121715,497.13,2050.472364,...,-1.259757,102.985700,103.668215,107.067400,105.118639,71.877481,NaN,0,NaN,0
1,2023-02-26,23157.07,23689.99,23059.18,23554.85,202323.73623,9506015.0,1.717748,630.81,1914.752280,...,-0.332698,103.010966,103.714559,107.065650,105.115965,67.646349,NaN,0,NaN,0
2,2023-02-27,23554.85,23897.99,23106.77,23492.09,283706.08590,11754195.0,-0.266442,791.22,1769.264806,...,-0.399656,103.038500,103.752027,107.063550,105.111528,63.613624,NaN,0,NaN,0
3,2023-02-28,23492.09,23600.00,23020.97,23141.57,264140.99894,9568743.0,-1.492077,579.03,1600.487431,...,-0.719487,103.075900,103.795870,107.059750,105.109125,66.036431,NaN,0,NaN,0
4,2023-03-01,23141.57,24000.00,23020.03,23628.97,315287.41737,9390487.0,2.106167,979.97,1426.661803,...,-1.375031,103.100700,103.822698,107.052466,105.102865,57.935523,NaN,0,NaN,0


# Agrupar variables

In [3]:
sufijos = [
    "_binance",
    "_fg",
    "_gtrends",
    "_hashrate",
    "_mDbinance",
    "_sp500",
    "_nasdaq",
    "_dxy",
]

# Crear un diccionario para almacenar las columnas separadas
columnas_por_sufijo = {sufijo: [] for sufijo in sufijos}

# Recorrer las columnas y clasificarlas
for col in df.columns:
    for sufijo in sufijos:
        if col.endswith(sufijo):
            columnas_por_sufijo[sufijo].append(col)
            break

# Ahora tienes las columnas organizadas por sufijo
suma = 5 # timestamp, 4 columnas respuesta
for sufijo, columnas in columnas_por_sufijo.items():
    print(f"\nSufijo {sufijo} ({len(columnas)}):")
    print(columnas)
    suma += len(columnas)

assert suma == df.shape[1], f"Suma: {suma}, columnas:{df.shape[1]}"


Sufijo _binance (17):
['open_binance', 'high_binance', 'low_binance', 'close_binance', 'volumen_binance', 'trades_binance', 'var_diaria_binance', 'rango_dia_binance', 'volatilidad_50_binance', 'momentum_50_binance', 'volatilidad_200_binance', 'momentum_200_binance', 'SMA_50_binance', 'EMA_50_binance', 'SMA_200_binance', 'EMA_200_binance', 'RSI_binance']

Sufijo _fg (1):
['fear_greed_index_fg']

Sufijo _gtrends (1):
['bitcoin_gtrends']

Sufijo _hashrate (5):
['hashrate_hashrate', 'SMA_50_hashrate', 'SMA_200_hashrate', 'EMA_50_hashrate', 'EMA_200_hashrate']

Sufijo _mDbinance (17):
['open_mDbinance', 'high_mDbinance', 'low_mDbinance', 'close_mDbinance', 'volumen_mDbinance', 'trades_mDbinance', 'var_diaria_mDbinance', 'rango_dia_mDbinance', 'volatilidad_50_mDbinance', 'momentum_50_mDbinance', 'volatilidad_200_mDbinance', 'momentum_200_mDbinance', 'SMA_50_mDbinance', 'EMA_50_mDbinance', 'SMA_200_mDbinance', 'EMA_200_mDbinance', 'RSI_mDbinance']

Sufijo _sp500 (16):
['close_sp500', 'high_s

# Eliminación de variables redundantes o con información duplicada


## 1. Binance – precios OHLC
**Eliminar:**
- open_binance
- high_binance
- low_binance
- volumen_binance: se calcula close_binance * volume_binance

**Dejar:**
- close_binance
- volume_binance
- trades_binance
- rango_dia_binance: diferencia de high y low
- var_diaria_binance: cambio porcentual entre un valor y el valor inmediatamente anterior (en porcentaje)

In [4]:
df.drop(columns=['open_binance', 'high_binance', 'low_binance', 'volumen_binance'], inplace=True)

## 2. Binance - Medias móviles
Mantener una combinación de corto y largo plazo, evitando duplicar SMA y EMA para ambos.

**Eliminar:**
- SMA_50_binance
- EMA_200_binance

**Dejar:**
- EMA_50_binance
- SMA_200_binance

In [5]:
df.drop(columns=['SMA_50_binance', 'EMA_200_binance'], inplace=True)

## 3. Binance - Otras varibles
Conservaremos las variables de volatilidad, momentum y RSI

## 4. Indicadores
Conservaremos los indicadores: `fear_greed_index_fg` y `bitcoin_gtrends`

## 5. Hashrate
Conservaremos la variable `hashrate_hashrate` y mantendremos una combinación de corto y largo plazo para las medias móviles como en Binance

**Eliminar:**
- SMA_50_hashrate
- EMA_200_hashrate

**Dejar:**
- EMA_50_hashrate
- SMA_200_hashrate

In [6]:
df.drop(columns=['SMA_50_hashrate', 'EMA_200_hashrate'], inplace=True)

## 6. Mercado Derivados Binance, S&P 500, Nasdaq, DXY
Al compartir las mismas variables que Binance, repetiremos el proceso:

**Eliminar:**
- open
- high
- low
- SMA_50
- EMA_200

**Dejar:**
- close
- volume
- trades
- var_diaria
- rango dia
- volatilidad (50/200)
- momentum (50/200)
- SMA_200
- EMA_50

In [7]:
# mDbinance
df.drop(columns=[
    'open_mDbinance',
    'high_mDbinance',
    'low_mDbinance',
    'SMA_50_mDbinance',
    'EMA_200_mDbinance'
        ],
        inplace=True)

# S&P 500, Nasdaq, DXY
sufijos = ['_sp500', '_nasdaq', '_dxy']
columnas = ['open', 'high', 'low', 'SMA_50', 'EMA_200']
columnas_a_eliminar = [col + suf for col in columnas for suf in sufijos]

df.drop(columns=columnas_a_eliminar, inplace=True)

## 7. Varibles respuesta categóricas
Para mostrar la creación de la variable respuesta gráficamente se creó una variable categórica auxiliar que indicaba la tendencia alcista o bajista del bitcoin. Como no se va a realizar problemas de clasificación se eliminarán estas variables

In [8]:
df.drop(columns=['tendencia_futura_7d', 'tendencia_futura_30d'], inplace=True)

## 8. Conjunto de datos para 7 días
No se eliminará ninguna otra variable salvo la variable respuesta para 30 días

In [9]:
df_7d = df.drop(columns=['pct_cambio_30d'])

output_path = r"../../data/nuevos/dataset_7d_nuevos.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_7d.to_csv(output_path, index=False)

## 9. Conjunto de datos para 30 días
No se eliminará ninguna otra variable salvo la variable respuesta para 7 días

In [10]:
df_30d = df.drop(columns=['pct_cambio_7d'])

output_path = r"../../data/nuevos/dataset_30d_nuevos.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_30d.to_csv(output_path, index=False)

In [11]:
df.head()

,timestamp,close_binance,trades_binance,var_diaria_binance,rango_dia_binance,volatilidad_50_binance,momentum_50_binance,volatilidad_200_binance,momentum_200_binance,EMA_50_binance,...,rango_dia_dxy,volatilidad_50_dxy,momentum_50_dxy,volatilidad_200_dxy,momentum_200_dxy,EMA_50_dxy,SMA_200_dxy,RSI_dxy,pct_cambio_7d,pct_cambio_30d
0,2023-02-25,23157.07,9124568.0,-0.121715,497.13,2050.472364,36.614643,2505.631809,0.030756,22133.807461,...,0.870000,0.952449,1.107048,3.600751,-1.259757,103.668215,107.067400,71.877481,NaN,NaN
1,2023-02-26,23554.85,9506015.0,1.717748,630.81,1914.752280,39.019404,2502.424710,-1.666524,22189.534619,...,0.839999,0.984920,1.219591,3.601748,-0.332698,103.714559,107.065650,67.646349,NaN,NaN
2,2023-02-27,23492.09,11754195.0,-0.266442,791.22,1769.264806,37.157422,2498.901331,-1.847969,22240.615222,...,0.809998,1.011849,1.332773,3.603028,-0.399656,103.752027,107.063550,63.613624,NaN,NaN
3,2023-02-28,23141.57,9568743.0,-1.492077,579.03,1600.487431,34.714284,2488.657693,-5.171802,22275.946782,...,0.580002,1.044432,1.815537,3.604948,-0.719487,103.795870,107.059750,66.036431,NaN,NaN
4,2023-03-01,23628.97,9390487.0,2.106167,979.97,1426.661803,35.482086,2481.601170,-3.323912,22329.006516,...,1.000000,1.062966,1.201090,3.608698,-1.375031,103.822698,107.052466,57.935523,NaN,NaN
